# Response Quality and Operational Playbook

## Business question

Which complaint segments indicate weak response quality, and what concrete review queues should operations teams prioritize?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from cx_complaints.data.schema import normalize_complaints
from cx_complaints.features.text_features import add_text_features
from cx_complaints.modelling.risk import score_complaint_risk
from cx_complaints.preprocessing.text import add_clean_text_column

plt.style.use("ggplot")

In [ ]:
raw_path = Path("data/raw/cfpb_sample.csv")
if not raw_path.exists():
    raise FileNotFoundError(
        "Expected data/raw/cfpb_sample.csv. Run the fetch command first."
    )

raw = pd.read_csv(raw_path)
complaints = normalize_complaints(raw)
complaints = add_clean_text_column(complaints)
complaints = add_text_features(complaints)
complaints = score_complaint_risk(complaints)
complaints.shape

## 1. Response quality segmentation

In [ ]:
response_segment = (
    complaints.assign(
        timely_clean=complaints["timely"].fillna("Unknown").astype(str).str.title(),
        response_clean=complaints["company_response"].fillna("Unknown").astype(str).str.title(),
    )
    .groupby(["timely_clean", "response_clean"], dropna=False)
    .agg(
        complaints=("complaint_id", "size"),
        avg_risk_score=("risk_score", "mean"),
        high_risk_share_pct=("risk_level", lambda s: (s == "high").mean() * 100),
    )
    .reset_index()
    .sort_values(["complaints", "avg_risk_score"], ascending=[False, False])
)
response_segment["avg_risk_score"] = response_segment["avg_risk_score"].round(2)
response_segment["high_risk_share_pct"] = response_segment["high_risk_share_pct"].round(1)
response_segment.head(20)

## 2. Priority operational queue

In [ ]:
queue = complaints.copy()
queue["timely_is_yes"] = queue["timely"].fillna("").str.lower() == "yes"
queue["weak_response"] = queue["company_response"].fillna("").str.lower().isin(
    ["closed", "closed with explanation", "in progress"]
)

priority_queue = queue[
    (queue["risk_level"] == "high")
    & (~queue["timely_is_yes"])
    & (queue["weak_response"])
].copy()
priority_queue = priority_queue.sort_values("risk_score", ascending=False)

priority_queue[
    [
        "complaint_id",
        "date_received",
        "company",
        "product",
        "issue",
        "company_response",
        "timely",
        "risk_score",
    ]
].head(25)

In [ ]:
kpi = pd.DataFrame(
    {
        "metric": [
            "Total complaints",
            "High-risk complaints",
            "Priority queue complaints",
        ],
        "count": [
            len(queue),
            int((queue["risk_level"] == "high").sum()),
            len(priority_queue),
        ],
    }
)
kpi["share_pct"] = (kpi["count"] / max(len(queue), 1) * 100).round(2)
kpi

## 3. Company and issue concentration inside priority queue

In [ ]:
queue_company = (
    priority_queue.groupby("company", dropna=False)
    .size()
    .rename("complaints")
    .reset_index()
    .sort_values("complaints", ascending=False)
)
queue_issue = (
    priority_queue.groupby("issue", dropna=False)
    .size()
    .rename("complaints")
    .reset_index()
    .sort_values("complaints", ascending=False)
)

queue_company.head(15), queue_issue.head(15)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_company = queue_company.head(10).iloc[::-1]
top_issue = queue_issue.head(10).iloc[::-1]

axes[0].barh(top_company["company"].fillna("Unknown"), top_company["complaints"], color="#6D597A")
axes[0].set_title("Priority Queue by Company")
axes[0].set_xlabel("Complaint count")

axes[1].barh(top_issue["issue"].fillna("Unknown"), top_issue["complaints"], color="#E56B6F")
axes[1].set_title("Priority Queue by Issue")
axes[1].set_xlabel("Complaint count")

fig.tight_layout()

## 4. Operational playbook

Suggested routine after running this notebook:

1. Daily: review top `risk_score` rows from `priority_queue`.
2. Weekly: track queue volume by company and issue for escalation triggers.
3. Monthly: compare queue composition with topic trend notebook outputs.

In [ ]:
# Optional export
output_dir = Path("reports/notebook")
output_dir.mkdir(parents=True, exist_ok=True)

priority_queue.to_csv(output_dir / "notebook_priority_queue.csv", index=False)
queue_company.to_csv(output_dir / "notebook_priority_queue_company.csv", index=False)
queue_issue.to_csv(output_dir / "notebook_priority_queue_issue.csv", index=False)
kpi.to_csv(output_dir / "notebook_priority_queue_kpi.csv", index=False)

output_dir